In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

from funnel_tools import (
    first_step, matched_events, next_step, prepare_events,
    read_events, summarise_steps,
)

rules = json.loads(Path("funnel_rules.json").read_text(encoding="utf-8"))
followup_hours = rules["followup"]["hours"]

In [ ]:
example_raw = read_events("worked_events.csv")
example_events, example_audit = prepare_events(example_raw, "event_id")
display(example_audit, example_events)

In [ ]:
room_list = first_step(example_events, "room_list_opened", "2026-07-06", "2026-07-07")
room = next_step(example_events, room_list, "room_chosen", followup_hours)
booking = next_step(example_events, room, "booking_started", followup_hours)
submission = next_step(example_events, booking, "booking_submitted", followup_hours)
example_steps = {
    "room_list_opened": room_list,
    "room_chosen": room,
    "booking_started": booking,
    "booking_submitted": submission,
}
example_paths = matched_events(example_steps)
display(example_paths)

In [ ]:
example_counts = summarise_steps(example_steps)
pd.testing.assert_frame_equal(example_counts, pd.read_csv("worked_counts.csv"))
expected_paths = (
    pd.read_csv("worked_paths.csv", keep_default_na=False)
    .set_index("user_id")
    .rename_axis("event_name", axis="columns")
)
actual_paths = (
    example_paths.pivot(index="user_id", columns="event_name", values="event_id")
    .reindex(index=expected_paths.index, columns=expected_paths.columns)
    .fillna("")
)
pd.testing.assert_frame_equal(actual_paths, expected_paths)
display(example_counts)

In [ ]:
project_raw = read_events("app_events.csv")
spot_check = pd.read_csv("spot_check_sample.csv", keep_default_na=False)
display(pd.read_csv("event_catalogue.csv"))
display(project_raw.loc[project_raw["user_id"].isin(spot_check["user_id"])])

In [ ]:
cohort_start = ""
cohort_end = ""
duplicate_rule = ""
focus_segment = ""
event_names = []
export_csv = False
count_in_notebook = True

In [ ]:
project_steps = {}
if event_names and cohort_start and cohort_end and duplicate_rule and focus_segment:
    valid_windows = {
        (window["start_inclusive"], window["end_exclusive"])
        for window in rules["candidate_cohort_windows_utc"]
    }
    assert (cohort_start, cohort_end) in valid_windows
    assert focus_segment in rules["segments"]
    assert event_names == [step["event_name"] for step in rules["funnel_steps"]]
    project_events, project_audit = prepare_events(project_raw, duplicate_rule)
    project_steps[event_names[0]] = first_step(
        project_events, event_names[0], cohort_start, cohort_end
    )
    # TODO: add the next three eligible sets using next_step.
    display(project_audit)

In [ ]:
if event_names and list(project_steps) == event_names:
    project_paths = matched_events(project_steps)
    display(project_paths.loc[project_paths["user_id"].isin(spot_check["user_id"])])
    if export_csv:
        project_paths.to_csv("project_eligible.csv", index=False)
    if count_in_notebook:
        project_counts = summarise_steps(project_steps)
        display(project_counts.loc[project_counts["group"].isin(["overall", focus_segment])])
        # TODO: reconcile segment counts and compare the sample with its source histories.
        if export_csv:
            project_counts.to_csv("project_funnel_counts.csv", index=False)